# 01 · Data ingestion and schema validation

**Goal:** confirm that the seven handover datasets are available, profile each one (size, rows, columns, dtypes, battery groups, date ranges, missing values) and validate `DATA_DICTIONARY.csv` against the columns actually observed.

**Data location.** The CSVs are *not* in Git. They are read from `DATA_ROOT` (environment variable, default `data/` relative to the repository root):

```bash
export DATA_ROOT=/mnt/project-data   # or copy the CSVs into ./data/
jupyter nbconvert --to notebook --execute notebooks/01_data_ingestion.ipynb --output-dir /tmp
```

Nothing in this notebook creates, imputes or replaces data. If a file is missing the notebook stops with an explicit error.

## 1. Setup and path resolution

In [ ]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd

# Resolve the repository root whether the kernel starts in the repo root or in notebooks/.
REPO_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src" / "data_pipeline.py").is_file())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src import data_pipeline as dp

DATA_ROOT = dp.resolve_data_root(os.environ.get("DATA_ROOT", "data"), REPO_ROOT)
RESULTS_DIR = REPO_ROOT / "results"

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)
print("DATA_ROOT env :", os.environ.get("DATA_ROOT", "<unset, default 'data'>"))
print("DATA_ROOT dir exists:", DATA_ROOT.is_dir())

## 2. File inventory

Check that `DATA_ROOT` exists and that every expected file is present. `check_data_files` raises `DataFilesMissingError` listing the missing files and how to provide them; no fallback data is generated.

In [ ]:
inventory = dp.check_data_files(DATA_ROOT)  # all seven files are required here
inventory

## 3. Profile the cycle- and RPT-level files

These files are small enough (cycle-level / RPT-level) to load fully. `Cycle_Date` and `Test_Date` are parsed as datetimes where present, `low_memory=False` keeps dtype inference consistent, and rows are kept in file order (no shuffling).

In [ ]:
frames: dict[str, pd.DataFrame] = {}
for name in dp.EXPECTED_FILES:
    if name in dp.LARGE_FILES:
        continue
    df = dp.read_dataset(DATA_ROOT / name)
    frames[name] = df
    print(f"=== {name}  shape={df.shape}  size={inventory.set_index('file').loc[name, 'size_mb']} MB")
    print("columns:", list(df.columns))
    display(dp.dtype_summary(df))

## 4. Profile the point-level file in chunks

`cleaned_battery_data.csv` is point-level and can be large, so it is streamed with `chunksize`. Only the header is read up front; counts, per-battery row totals, null counts and date ranges are accumulated chunk by chunk. Its exact schema is not documented in the dictionary, so it is reported as observed.

In [ ]:
POINT_FILE = "cleaned_battery_data.csv"
point_profile = dp.profile_csv_chunked(DATA_ROOT / POINT_FILE, chunksize=500_000)
print(f"{POINT_FILE}: rows={point_profile['rows']:,}  columns={len(point_profile['columns'])}")
print("columns:", point_profile["columns"])
display(pd.DataFrame({"dtype": point_profile["dtypes"], "missing": point_profile["null_counts"]}))
display(point_profile["rows_per_group"].rename("rows").to_frame())
print("date ranges:", point_profile["date_ranges"] or "no Cycle_Date/Test_Date column in this file")

## 5. Row counts and file overview

In [ ]:
overview = pd.DataFrame(
    [{"file": n, "rows": len(df), "columns": df.shape[1]} for n, df in frames.items()]
    + [{"file": POINT_FILE, "rows": point_profile["rows"], "columns": len(point_profile["columns"])}]
).merge(inventory[["file", "size_mb"]], on="file").set_index("file")
overview

## 6. Battery groups

Every file carrying `Battery_ID` should contain exactly the six groups **g1, v4, v5, w8, w9, w10**.

In [ ]:
group_rows = {}
group_checks = {}
for name, df in frames.items():
    if dp.GROUP_COL in df:
        group_rows[name] = df[dp.GROUP_COL].astype(str).value_counts()
        group_checks[name] = dp.check_battery_groups(df[dp.GROUP_COL])
if len(point_profile["rows_per_group"]):
    group_rows[POINT_FILE] = point_profile["rows_per_group"]
    group_checks[POINT_FILE] = dp.check_battery_groups(point_profile["rows_per_group"].index)

display(pd.DataFrame(group_rows).reindex(sorted(set().union(*[s.index for s in group_rows.values()]))).fillna(0).astype(int))
battery_group_check = pd.DataFrame(group_checks).T[["ok", "missing", "unexpected"]]
battery_group_check

## 7. Date ranges per battery

`Cycle_Date` (cycling) and `Test_Date` (RPT) ranges per battery, including unparseable/missing dates (`n_missing`).

In [ ]:
for name, df in frames.items():
    for col in dp.DATE_COLUMNS:
        if col in df and dp.GROUP_COL in df:
            print(f"--- {name} · {col}")
            display(dp.date_ranges(df, col))

## 8. Missing values

Counts per file and column (only columns with at least one missing value). Missing temperatures and pre-RPT SOH are expected and are **reported, not filled**.

In [ ]:
missing = pd.concat(
    {name: df.isna().sum() for name, df in frames.items() if name != "DATA_DICTIONARY.csv"}
    | {POINT_FILE: point_profile["null_counts"]},
    axis=1,
)
missing = missing.loc[missing.fillna(0).sum(axis=1) > 0]
missing if len(missing) else "No missing values in any file."

## 9. Validate the data dictionary against observed columns

For every column: is it documented in `DATA_DICTIONARY.csv`, and in which files does it appear? The README states the dictionary covers 28 columns; the actual count is reported below rather than assumed.

In [ ]:
dd = dp.load_data_dictionary(DATA_ROOT / "DATA_DICTIONARY.csv")
print("dictionary name column:", dd.attrs["name_column"], "| documented columns:", dd["column_name"].nunique())
display(dd)

observed = {n: list(df.columns) for n, df in frames.items() if n != "DATA_DICTIONARY.csv"}
observed[POINT_FILE] = point_profile["columns"]
dict_vs_files = dp.compare_dictionary(dd["column_name"], observed)
dict_vs_files

In [ ]:
cycle_level = ["final_ml_dataset.csv", "ml_training_dataset.csv", "final_battery_cycling_features.csv"]
documented_not_observed = dict_vs_files.index[dict_vs_files["in_dictionary"] & (dict_vs_files["n_files"] == 0)].tolist()
undocumented_cycle_cols = dict_vs_files.index[
    ~dict_vs_files["in_dictionary"] & dict_vs_files[cycle_level].any(axis=1)
].tolist()
undocumented_other = dict_vs_files.index[
    ~dict_vs_files["in_dictionary"] & ~dict_vs_files[cycle_level].any(axis=1)
].tolist()
final_ml_matches_dict = set(observed["final_ml_dataset.csv"]) == set(dd["column_name"])
same_schema_train_final = list(observed["ml_training_dataset.csv"]) == list(observed["final_ml_dataset.csv"])

print("Documented but not observed in any file :", documented_not_observed or "none")
print("Undocumented columns in cycle-level files:", undocumented_cycle_cols or "none")
print("Undocumented columns in other files      :", undocumented_other or "none")
print("final_ml_dataset columns == dictionary   :", final_ml_matches_dict)
print("ml_training and final_ml share schema    :", same_schema_train_final)

## 10. Summary

In [ ]:
summary = pd.DataFrame(
    [
        ("All 7 expected files present", bool(inventory["present"].all()), f"DATA_ROOT exists: {DATA_ROOT.is_dir()}"),
        ("Battery groups == g1,v4,v5,w8,w9,w10 in every file", bool(battery_group_check["ok"].all()),
         "; ".join(f"{k}: missing={v['missing']} unexpected={v['unexpected']}" for k, v in group_checks.items() if not v["ok"]) or "all ok"),
        ("Dictionary documents all cycle-level columns", not undocumented_cycle_cols, str(undocumented_cycle_cols)),
        ("Every documented column is observed", not documented_not_observed, str(documented_not_observed)),
        ("ml_training / final_ml share the same schema", same_schema_train_final, ""),
        ("Documented column count", dd["column_name"].nunique() == 28, f"{dd['column_name'].nunique()} documented (README says 28)"),
        ("Missing temperature cells (final_ml_dataset)",
         int(frames["final_ml_dataset.csv"][[c for c in dp.TEMPERATURE_COLUMNS if c in frames["final_ml_dataset.csv"]]].isna().sum().sum()),
         "reported only; see notebook 02"),
    ],
    columns=["check", "result", "detail"],
)
summary

### Notes and unresolved assumptions

* **Point-level schema.** `cleaned_battery_data.csv` is not described by the data dictionary; its columns are reported as observed above and it is only profiled (chunked), not used for modelling yet.
* **Dictionary size.** README says 28 documented columns; the README's own table lists 27. The observed count is printed above — any gap should be resolved with the data owner.
* **Missing values** (temperatures, pre-RPT SOH in `final_ml_dataset.csv`) are measured here and analysed in notebook 02; none are imputed.
* **Date parsing.** `Cycle_Date`/`Test_Date` are parsed by pandas' default parser. `n_missing > 0` in section 7 indicates unparseable or absent timestamps that must be investigated before temporal alignment is trusted.